## **Aim**
To implement a program that creates a complete digital forensic case report containing evidence details, hash values, findings, conclusions, and recommendations.

## **Algorithm**
**Step 1:** Import `json`, `datetime`, `hashlib`, and `uuid` libraries.

**Step 2:** Define a `CaseReport` class with sections for:
   - Case metadata (ID, investigator, dates, classification)
   - Evidence inventory (items, hashes, chain of custody)
   - Findings (categorized by type with severity)
   - Analysis summary (timeline, root cause, impact)
   - Conclusions
   - Recommendations

**Step 3:** Implement methods to add evidence with automatic hash calculation.

**Step 4:** Implement methods to add findings with MITRE ATT&CK mapping.

**Step 5:** Generate comprehensive report in multiple formats (JSON, text, HTML).

**Step 6:** Include digital signature placeholder for report integrity.

In [1]:
import json
import hashlib
import uuid
from datetime import datetime
from collections import defaultdict

class CaseReport:
    def __init__(self, case_id, investigator, organization, classification="CONFIDENTIAL"):
        self.case_id = case_id
        self.report_id = str(uuid.uuid4())
        self.classification = classification
        self.status = "DRAFT"
        self.investigator = investigator
        self.organization = organization
        self.date_opened = None
        self.date_closed = None
        self.evidence = []
        self.findings = []
        self.analysis_summary = {}
        self.conclusions = []
        self.recommendations = []
        
    def add_evidence(self, evidence_id, evidence_type, description, file_path=None,
                   collected_by=None, location=None):
        hashes = {}
        if file_path:
            try:
                with open(file_path, "rb") as f:
                    content = f.read()
                hashes["SHA-256"] = hashlib.sha256(content).hexdigest()
                hashes["MD5"] = hashlib.md5(content).hexdigest()
                hashes["SHA-1"] = hashlib.sha1(content).hexdigest()
            except:
                hashes["SHA-256"] = "FILE_NOT_FOUND"
                hashes["MD5"] = "FILE_NOT_FOUND"
                hashes["SHA-1"] = "FILE_NOT_FOUND"
        else:
            # Generate placeholder hashes for simulation
            hashes["SHA-256"] = hashlib.sha256(f"{evidence_id}{description}".encode()).hexdigest()
            hashes["MD5"] = hashlib.md5(f"{evidence_id}{description}".encode()).hexdigest()
            hashes["SHA-1"] = hashlib.sha1(f"{evidence_id}{description}".encode()).hexdigest()
        
        evidence = {
            "evidence_id": evidence_id,
            "type": evidence_type,
            "description": description,
            "hashes": hashes,
            "collected": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            "custodian": collected_by,
            "location": location,
            "chain_of_custody": 1
        }
        self.evidence.append(evidence)
        return evidence_id
    
    def add_finding(self, finding_id, severity, category, description, evidence_refs, iocs, mitre_technique=None):
        finding = {
            "finding_id": finding_id,
            "severity": severity,
            "category": category,
            "description": description,
            "evidence_refs": evidence_refs,
            "iocs": iocs,
            "mitre_technique": mitre_technique,
            "timestamp": datetime.now().isoformat()
        }
        self.findings.append(finding)
        return finding_id
    
    def set_analysis_summary(self, timeline, initial_access, threat_actor, c2, compromised_hosts,
                           compromised_accounts, data_exfiltrated, files_encrypted, ransomware, ransom_demand, recovery_status):
        self.analysis_summary = {
            "timeline": timeline,
            "initial_access": initial_access,
            "threat_actor": threat_actor,
            "c2_infrastructure": c2,
            "compromised_hosts": compromised_hosts,
            "compromised_accounts": compromised_accounts,
            "data_exfiltrated": data_exfiltrated,
            "files_encrypted": files_encrypted,
            "ransomware_variant": ransomware,
            "ransom_demand": ransom_demand,
            "recovery_status": recovery_status
        }
    
    def add_conclusion(self, conclusion):
        self.conclusions.append(conclusion)
    
    def add_recommendation(self, recommendation, priority="HIGH"):
        self.recommendations.append({"text": recommendation, "priority": priority})
    
    def finalize(self):
        self.status = "FINAL"
        self.date_closed = datetime.now().strftime('%Y-%m-%d')
    
    def generate_text_report(self):
        lines = []
        lines.append(f"{'='*70}")
        lines.append(f"DIGITAL FORENSIC CASE REPORT")
        lines.append(f"{'='*70}")
        lines.append(f"Case ID: {self.case_id}")
        lines.append(f"Report ID: {self.report_id}")
        lines.append(f"Classification: {self.classification}")
        lines.append(f"Status: {self.status}")
        lines.append(f"Investigator: {self.investigator}")
        lines.append(f"Organization: {self.organization}")
        if self.date_opened:
            lines.append(f"Date Opened: {self.date_opened}")
        if self.date_closed:
            lines.append(f"Date Closed: {self.date_closed}")
        lines.append(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        lines.append(f"")
        
        # Evidence Inventory
        lines.append(f"{'='*70}")
        lines.append(f"EVIDENCE INVENTORY")
        lines.append(f"{'='*70}")
        lines.append(f"Total Evidence Items: {len(self.evidence)}")
        lines.append(f"")
        
        for ev in self.evidence:
            lines.append(f"{ev['evidence_id']}: {ev.get('description', 'No description')}")
            lines.append(f"  Type: {ev['type']}")
            lines.append(f"  Description: {ev['description']}")
            for h_type, h_val in ev['hashes'].items():
                lines.append(f"  {h_type}: {h_val}")
            lines.append(f"  Collected: {ev['collected']}")
            lines.append(f"  Custodian: {ev['custodian']}")
            lines.append(f"  Location: {ev['location']}")
            lines.append(f"  Chain of Custody: {ev['chain_of_custody']} transfers")
            lines.append(f"")
        
        # Findings
        lines.append(f"{'='*70}")
        lines.append(f"FINDINGS")
        lines.append(f"{'='*70}")
        lines.append(f"Total Findings: {len(self.findings)}")
        lines.append(f"")
        
        for f in self.findings:
            mitre = f" ({f['mitre_technique']})" if f['mitre_technique'] else ""
            lines.append(f"{f['finding_id']} [{f['severity']}] - {f['category']}{mitre}")
            lines.append(f"  Description: {f['description']}")
            lines.append(f"  Evidence: {', '.join(f['evidence_refs'])}")
            if f['iocs']:
                lines.append(f"  IOCs: {', '.join(f['iocs'])}")
            lines.append(f"")
        
        # Analysis Summary
        if self.analysis_summary:
            lines.append(f"{'='*70}")
            lines.append(f"ANALYSIS SUMMARY")
            lines.append(f"{'='*70}")
            for k, v in self.analysis_summary.items():
                lines.append(f"{k.replace('_', ' ').title()}: {v}")
            lines.append(f"")
        
        # Conclusions
        lines.append(f"{'='*70}")
        lines.append(f"CONCLUSIONS")
        lines.append(f"{'='*70}")
        for i, c in enumerate(self.conclusions, 1):
            lines.append(f"{i}. {c}")
        lines.append(f"")
        
        # Recommendations
        lines.append(f"{'='*70}")
        lines.append(f"RECOMMENDATIONS")
        lines.append(f"{'='*70}")
        
        high_recs = [r for r in self.recommendations if r['priority'] == 'HIGH']
        med_recs = [r for r in self.recommendations if r['priority'] == 'MEDIUM']
        low_recs = [r for r in self.recommendations if r['priority'] == 'LOW']
        
        if high_recs:
            lines.append(f"IMMEDIATE ACTIONS:")
            for i, r in enumerate(high_recs, 1):
                lines.append(f"{i}. {r['text']}")
            lines.append(f"")
        
        if med_recs:
            lines.append(f"SHORT-TERM IMPROVEMENTS:")
            for i, r in enumerate(med_recs, 1):
                lines.append(f"{i}. {r['text']}")
            lines.append(f"")
        
        if low_recs:
            lines.append(f"LONG-TERM STRATEGIC:")
            for i, r in enumerate(low_recs, 1):
                lines.append(f"{i}. {r['text']}")
            lines.append(f"")
        
        # Report Integrity
        report_content = "\n".join(lines)
        report_hash = hashlib.sha256(report_content.encode()).hexdigest()
        
        lines.append(f"{'='*70}")
        lines.append(f"REPORT INTEGRITY")
        lines.append(f"{'='*70}")
        lines.append(f"Report SHA-256: {report_hash}")
        lines.append(f"Digital Signature: [PENDING - To be signed by Case Lead]")
        lines.append(f"Report Version: 1.0")
        lines.append(f"Next Review Date: {(datetime.now().replace(month=datetime.now().month+1)).strftime('%Y-%m-%d')}")
        lines.append(f"")
        lines.append(f"--- END OF REPORT ---")
        
        return "\n".join(lines)
    
    def save_json(self, filepath):
        data = {
            "case_id": self.case_id,
            "report_id": self.report_id,
            "classification": self.classification,
            "status": self.status,
            "investigator": self.investigator,
            "organization": self.organization,
            "date_opened": self.date_opened,
            "date_closed": self.date_closed,
            "evidence": self.evidence,
            "findings": self.findings,
            "analysis_summary": self.analysis_summary,
            "conclusions": self.conclusions,
            "recommendations": self.recommendations,
            "generated": datetime.now().isoformat()
        }
        with open(filepath, "w") as f:
            json.dump(data, f, indent=2)

def main():
    # Create case report
    report = CaseReport(
        case_id="CASE-2026-001",
        investigator="Senior Forensic Analyst",
        organization="Cyber Forensics Unit",
        classification="CONFIDENTIAL"
    )
    report.date_opened = "2026-08-18"
    
    # Add evidence
    report.add_evidence(
        "EVD-001", "Disk Image",
        "Bit-stream image of compromised workstation HDD",
        collected_by="Evidence Technician A",
        location="Evidence Locker A-12"
    )
    
    report.add_evidence(
        "EVD-002", "Memory Image",
        "Volatile memory capture from WORKSTATION-01",
        collected_by="Forensic Analyst B",
        location="Evidence Locker A-12"
    )
    
    report.add_evidence(
        "EVD-003", "Network Capture",
        "Full packet capture from 2026-08-18 10:00-17:00",
        collected_by="Network Forensics Team",
        location="Evidence Locker A-13"
    )
    
    report.add_evidence(
        "EVD-004", "Email Artifact",
        "Original phishing email with malicious attachment",
        collected_by="Email Security Team",
        location="Evidence Locker A-13"
    )
    
    report.add_evidence(
        "EVD-005", "Malware Binary",
        "Extracted ransomware payload (BlackCat)",
        collected_by="Malware Analyst",
        location="Evidence Locker A-14"
    )
    
    report.add_evidence(
        "EVD-006", "Registry Artifacts",
        "Exported registry hives from compromised system",
        collected_by="Forensic Analyst C",
        location="Evidence Locker A-14"
    )
    
    report.add_evidence(
        "EVD-007", "Log Files",
        "Firewall logs covering attack timeframe",
        collected_by="Network Security Team",
        location="Evidence Locker A-13"
    )
    
    report.add_evidence(
        "EVD-008", "Volume Shadow Copy",
        "VSS snapshot before deletion by ransomware",
        collected_by="Forensic Analyst D",
        location="Evidence Locker A-15"
    )
    
    # Add findings
    report.add_finding(
        "FND-001", "CRITICAL", "INITIAL_ACCESS",
        "Phishing email with malicious macro document delivered to user@company.com",
        ["EVD-004"],
        ["paypal-security-update.tk", "Invoice_2026-0820.docx"],
        "T1566.001"
    )
    
    report.add_finding(
        "FND-002", "CRITICAL", "EXECUTION",
        "Malicious macro executed PowerShell to download Cobalt Strike beacon",
        ["EVD-001", "EVD-002"],
        ["http://192.168.100.50/beacon.ps1"],
        "T1059.001"
    )
    
    report.add_finding(
        "FND-003", "HIGH", "PERSISTENCE",
        "Beacon established persistence via HKCU Run key and scheduled task",
        ["EVD-001", "EVD-006"],
        ["HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run"],
        "T1547.001"
    )
    
    report.add_finding(
        "FND-004", "CRITICAL", "PRIVILEGE_ESCALATION",
        "Beacon impersonated SYSTEM token via named pipe",
        ["EVD-001", "EVD-002"],
        ["Named pipe impersonation"],
        "T1134.001"
    )
    
    report.add_finding(
        "FND-005", "CRITICAL", "DEFENSE_EVASION",
        "Windows Defender real-time protection disabled",
        ["EVD-001", "EVD-002", "EVD-006"],
        ["Set-MpPreference -DisableRealtimeMonitoring $true"],
        "T1562.001"
    )
    
    report.add_finding(
        "FND-006", "HIGH", "DEFENSE_EVASION",
        "Security and System event logs cleared",
        ["EVD-001", "EVD-002"],
        ["wevtutil cl Security && wevtutil cl System"],
        "T1070.001"
    )
    
    report.add_finding(
        "FND-007", "CRITICAL", "CREDENTIAL_ACCESS",
        "LSASS memory dumped using comsvcs.dll",
        ["EVD-001", "EVD-002"],
        ["rundll32.exe comsvcs.dll MiniDump"],
        "T1003.001"
    )
    
    report.add_finding(
        "FND-008", "HIGH", "DISCOVERY",
        "Internal network scan and file share enumeration",
        ["EVD-003", "EVD-007"],
        ["192.168.1.0/24", "\\FILESERVER\\Data", "\\FILESERVER\\HR", "\\FILESERVER\\Finance"],
        "T1018, T1083"
    )
    
    report.add_finding(
        "FND-009", "CRITICAL", "LATERAL_MOVEMENT",
        "Lateral movement to FILESERVER and DC01",
        ["EVD-001", "EVD-003", "EVD-007"],
        ["PsExec", "aad3b435b51404eeaad3b435b51404ee:31d6cfe0d16ae931b73c59d7e0c089c0"],
        "T1021.002, T1021.004"
    )
    
    report.add_finding(
        "FND-010", "CRITICAL", "COLLECTION",
        "12.5 GB of sensitive data staged for exfiltration",
        ["EVD-001", "EVD-002"],
        ["C:\\Temp\\staging\\exfil_20260820.zip"],
        "T1560.001"
    )
    
    report.add_finding(
        "FND-011", "CRITICAL", "IMPACT",
        "BlackCat ransomware encryption and shadow copy deletion",
        ["EVD-001", "EVD-002", "EVD-005", "EVD-008"],
        [".blackcat extension", "vssadmin delete shadows /all /quiet"],
        "T1486, T1490"
    )
    
    # Analysis summary
    report.set_analysis_summary(
        timeline="2026-08-18 10:30 to 2026-08-18 17:15 (6 hours 45 minutes)",
        initial_access="Phishing email with malicious macro document",
        threat_actor="Likely BlackCat/ALPHV ransomware affiliate",
        c2="192.168.100.50 (Cobalt Strike)",
        compromised_hosts="3 (WORKSTATION-01, FILESERVER, DC01)",
        compromised_accounts="2 (user, DOMAIN\\admin)",
        data_exfiltrated="12.5 GB (HR, Finance, Source Code)",
        files_encrypted="15,000+ (.blackcat extension)",
        ransomware="BlackCat/ALPHV",
        ransom_demand="1 BTC",
        recovery_status="Partial (VSS available for some systems)"
    )
    
    # Conclusions
    conclusions = [
        "The incident began with a targeted phishing email containing a malicious macro-enabled document.",
        "The attacker used Cobalt Strike for command and control, establishing persistence within 20 minutes.",
        "Privilege escalation to SYSTEM was achieved via token manipulation within 1 hour.",
        "Defense evasion techniques included disabling Windows Defender and clearing event logs.",
        "Credential access via LSASS dump enabled lateral movement across the network.",
        "The attacker conducted reconnaissance, discovering file shares on multiple servers.",
        "Lateral movement to FILESERVER and DC01 used PsExec and Pass-the-Hash techniques.",
        "12.5 GB of sensitive data was collected and exfiltrated before encryption.",
        "BlackCat ransomware was deployed, encrypting 15,000+ files and deleting volume shadow copies.",
        "The attack demonstrates a mature, well-resourced threat actor following a standard ransomware kill chain."
    ]
    for c in conclusions:
        report.add_conclusion(c)
    
    # Recommendations
    high_recs = [
        "Isolate all compromised systems from the network immediately.",
        "Reset passwords for all compromised accounts (user, DOMAIN\\admin, service accounts).",
        "Restore encrypted files from verified clean backups; verify backup integrity before restore.",
        "Rebuild compromised systems from known good images; do not attempt to clean.",
        "Rotate all encryption keys and certificates potentially accessed during the breach.",
        "Block C2 IP (192.168.100.50) and associated domains at network perimeter."
    ]
    for r in high_recs:
        report.add_recommendation(r, "HIGH")
    
    med_recs = [
        "Implement application control (AppLocker/WDAC) to prevent unauthorized macro execution.",
        "Deploy EDR with behavioral detection for PowerShell, WMI, and process injection.",
        "Enable LSASS protection (RunAsPPL) and Credential Guard.",
        "Implement network segmentation to limit lateral movement.",
        "Deploy canary tokens/honeypots in sensitive directories for early detection.",
        "Conduct phishing simulation and awareness training for all employees."
    ]
    for r in med_recs:
        report.add_recommendation(r, "MEDIUM")
    
    low_recs = [
        "Implement Zero Trust architecture with micro-segmentation.",
        "Deploy automated threat hunting for TTPs observed in this incident.",
        "Establish 24/7 SOC with incident response playbooks for ransomware.",
        "Implement immutable backup strategy with offline/air-gapped copies.",
        "Conduct regular purple team exercises simulating BlackCat TTPs.",
        "Review and update incident response plan based on lessons learned.",
        "Implement data loss prevention (DLP) for sensitive data classification.",
        "Establish threat intelligence sharing with industry ISACs."
    ]
    for r in low_recs:
        report.add_recommendation(r, "LOW")
    
    # Finalize and generate
    report.finalize()
    text_report = report.generate_text_report()
    
    # Save to files
    with open("case_report.txt", "w") as f:
        f.write(text_report)
    
    report.save_json("case_report.json")
    
    print("Generating complete digital forensic case report...")
    print(text_report)
    print(f"\nReport saved to case_report.txt and case_report.json")

if __name__ == "__main__":
    main()

Generating complete digital forensic case report...
DIGITAL FORENSIC CASE REPORT
Case ID: CASE-2026-001
Report ID: 50296c94-2580-4602-849d-b685d5ddb191
Classification: CONFIDENTIAL
Status: FINAL
Investigator: Senior Forensic Analyst
Organization: Cyber Forensics Unit
Date Opened: 2026-08-18
Date Closed: 2026-08-20
Report Generated: 2026-08-20 10:19:30

EVIDENCE INVENTORY
Total Evidence Items: 8

EVD-001: Bit-stream image of compromised workstation HDD
  Type: Disk Image
  Description: Bit-stream image of compromised workstation HDD
  SHA-256: 4c6223d56d7ec18b0bb0e3342bce3e5be7f986918876120353a4b1f547a88c51
  MD5: 464bfab0a0a7789ed44e073451e918c8
  SHA-1: 16c65c827a4c5fb38b1876fe980352a532dc2b78
  Collected: 2026-08-20 10:19:30
  Custodian: Evidence Technician A
  Location: Evidence Locker A-12
  Chain of Custody: 1 transfers

EVD-002: Volatile memory capture from WORKSTATION-01
  Type: Memory Image
  Description: Volatile memory capture from WORKSTATION-01
  SHA-256: 57f531d5ca332b2e8c

## **Result**
This the program successfully creates a complete digital forensic case report containing evidence details, hash values, findings, conclusions, and recommendations.